### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 DWT](#31-dwt-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
import pywt as pwt
from pywt import wavedec

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold, cross_val_score, cross_validate
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

DWT features are extracted from each 20s window decomposed to level 5. For each channel, energy, variance, standard deviation, and peak amplitude are computed across all coefficient arrays, alongside hemispheric asymmetry features for each frontal, frontotemporal, and temporal channel pair, yielding 198 features per window.

##### 3.1 DWT Feature Extraction

In [4]:
CHANNELS = ['F3', 'F4', 'FT7', 'FT8', 'T7', 'T8']
ASYM_PAIRS = [(0, 1), (2, 3), (4, 5)]  # F3/F4, FT7/FT8, T7/T8

def extract_dwt_features(segmented_windows, labels):
    features = []
    for window in segmented_windows:
        channel_features = []
        channel_coeffs = []

        # Per-channel per-band stats
        for ch in range(window.shape[0]):
            coeffs = wavedec(window[ch], 'db4', level=5)
            channel_coeffs.append(coeffs)
            for coeff in coeffs:
                channel_features.extend([
                    np.mean(coeff),
                    np.std(coeff),
                    np.var(coeff),
                    np.sum(coeff**2),        # band energy
                    np.max(np.abs(coeff)),   # peak amplitude
                ])

        for left, right in ASYM_PAIRS:
            for level in range(6):  # 6 coefficient arrays at level=5
                left_energy = np.sum(channel_coeffs[left][level]**2)
                right_energy = np.sum(channel_coeffs[right][level]**2)
                asymmetry = (left_energy - right_energy) / (left_energy + right_energy + 1e-8)
                channel_features.append(asymmetry)

        features.append(channel_features)

    X = np.array(features)
    y = np.array(labels)
    return X, y


##### 3.2 Class Distribution

In [5]:
X, y = extract_dwt_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 198)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are tuned once using Optuna with StratifiedGroupKFold (5 splits) on the full dataset, ensuring no subject appears in both train and validation folds. Best parameters are then frozen and used for final LOSO evaluation. Results are also evaluated using 10-fold CV for direct comparison with existing literature.

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [11]:
#Stops hyperparameter tuning if no improvement after a set # of trials
def no_improvement_callback(study, trial, n_trials_no_improve=10):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_en == 0)
pos_count = np.sum(y_en == 1)
scale_en = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_en,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_en, y_en, cv=cv, groups=groups_en, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Save best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_en
best_params['random_state'] = 42

[I 2026-03-02 12:01:08,270] A new study created in memory with name: no-name-e704bbb8-fbb4-4bdb-87be-2a2ac27c7444
[I 2026-03-02 12:01:12,555] Trial 0 finished with value: 0.49969823229142535 and parameters: {'n_estimators': 129, 'max_depth': 4, 'learning_rate': 0.04767378124445038, 'subsample': 0.9998321989510456, 'colsample_bytree': 0.6886574825305211, 'min_child_weight': 6, 'gamma': 2.020751716083664}. Best is trial 0 with value: 0.49969823229142535.
[I 2026-03-02 12:01:26,499] Trial 1 finished with value: 0.5095128979910881 and parameters: {'n_estimators': 487, 'max_depth': 6, 'learning_rate': 0.03314424081731788, 'subsample': 0.9208924077099111, 'colsample_bytree': 0.8862186773106301, 'min_child_weight': 3, 'gamma': 2.6702412677201415}. Best is trial 1 with value: 0.5095128979910881.
[I 2026-03-02 12:01:34,639] Trial 2 finished with value: 0.5063040522542326 and parameters: {'n_estimators': 173, 'max_depth': 6, 'learning_rate': 0.030054412800798087, 'subsample': 0.7890262027346452,

Best params: {'n_estimators': 205, 'max_depth': 8, 'learning_rate': 0.05118396892453772, 'subsample': 0.7600976279044747, 'colsample_bytree': 0.601501331403713, 'min_child_weight': 2, 'gamma': 2.950995953537096}
Best CV accuracy: 0.5204


In [12]:
# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
en_accuracies = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_en, y_en, groups_en)):
    X_train, X_test = X_en[train_idx], X_en[test_idx]
    y_train, y_test = y_en[train_idx], y_en[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    en_accuracies.append(accuracy_score(y_test, preds))
    print(f"Fold {fold+1} | Accuracy: {en_accuracies[-1]:.4f}")

print("\n=== LOSO XGB (EN) ===")
print(f"Accuracy:  {np.mean(en_accuracies):.4f} ± {np.std(en_accuracies):.4f}")


Fold 1 | Accuracy: 0.4865
Fold 2 | Accuracy: 0.4150
Fold 3 | Accuracy: 0.4571
Fold 4 | Accuracy: 0.5744
Fold 5 | Accuracy: 0.3836
Fold 6 | Accuracy: 0.7442
Fold 7 | Accuracy: 0.8316
Fold 8 | Accuracy: 0.6538
Fold 9 | Accuracy: 0.7143
Fold 10 | Accuracy: 0.4981
Fold 11 | Accuracy: 0.5634
Fold 12 | Accuracy: 0.4742
Fold 13 | Accuracy: 1.0000
Fold 14 | Accuracy: 0.6571
Fold 15 | Accuracy: 0.5093
Fold 16 | Accuracy: 0.5604
Fold 17 | Accuracy: 0.4740
Fold 18 | Accuracy: 0.4848
Fold 19 | Accuracy: 0.3478
Fold 20 | Accuracy: 0.5980
Fold 21 | Accuracy: 0.5776
Fold 22 | Accuracy: 0.5789
Fold 23 | Accuracy: 0.3613
Fold 24 | Accuracy: 0.1781
Fold 25 | Accuracy: 0.6420
Fold 26 | Accuracy: 0.4167
Fold 27 | Accuracy: 0.5636
Fold 28 | Accuracy: 0.5366
Fold 29 | Accuracy: 0.6290
Fold 30 | Accuracy: 0.5000
Fold 31 | Accuracy: 0.5512
Fold 32 | Accuracy: 0.5265
Fold 33 | Accuracy: 0.4853
Fold 34 | Accuracy: 0.5041

=== LOSO XGB (EN) ===
Accuracy:  0.5435 ± 0.1443


In [13]:
# 10 Fold Cross CV with same tuned params
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_10fold = cross_val_score(
    XGBClassifier(**best_params, n_jobs=-1),
    X_en, y_en,
    cv=cv_10fold,
    scoring='accuracy',
    n_jobs=-1
)

print("\n=== 10-FoldCV XGB (EN) ===")
print(f"Accuracy: {scores_10fold.mean():.4f} ± {scores_10fold.std():.4f}")


=== 10-FoldCV XGB (EN) ===
Accuracy: 0.7119 ± 0.0084


##### 5.1.2 Positive vs. Negative

In [14]:
#Stops hyperparameter tuning if no improvement after a set # of trials
def no_improvement_callback(study, trial, n_trials_no_improve=10):
    if trial.number >= n_trials_no_improve:
        recent_values = [t.value for t in study.trials[-n_trials_no_improve:]]
        if max(recent_values) <= study.best_value:
            study.stop()

# Class balancing
neg_count = np.sum(y_pn == 0)
pos_count = np.sum(y_pn == 1)
scale_pn = neg_count / pos_count

# Tune once on full dataset with StratifiedGroupKFold
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': scale_pn,
        'eval_metric':      'logloss',
    }
    model = XGBClassifier(**params, n_jobs=-1)
    cv = StratifiedGroupKFold(n_splits=5)
    scores = cross_val_score(model, X_pn, y_pn, cv=cv, groups=groups_pn, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, callbacks=[lambda s, t: no_improvement_callback(s, t)])
print(f"Best params: {study.best_params}")
print(f"Best CV accuracy: {study.best_value:.4f}")

# Save best params
best_params = study.best_params
best_params['scale_pos_weight'] = scale_pn
best_params['random_state'] = 42

[I 2026-03-02 12:06:38,180] A new study created in memory with name: no-name-a0c805dc-dea3-4318-a653-abfcea844dc3
[I 2026-03-02 12:06:45,719] Trial 0 finished with value: 0.5123983801562971 and parameters: {'n_estimators': 594, 'max_depth': 3, 'learning_rate': 0.05869131150464924, 'subsample': 0.6964942842955001, 'colsample_bytree': 0.8513450849685094, 'min_child_weight': 3, 'gamma': 2.5414158455995404}. Best is trial 0 with value: 0.5123983801562971.
[I 2026-03-02 12:06:52,835] Trial 1 finished with value: 0.4964946785882205 and parameters: {'n_estimators': 227, 'max_depth': 4, 'learning_rate': 0.020048578363091568, 'subsample': 0.9727716811944722, 'colsample_bytree': 0.827208319520826, 'min_child_weight': 2, 'gamma': 0.9732850747591831}. Best is trial 0 with value: 0.5123983801562971.
[I 2026-03-02 12:06:56,759] Trial 2 finished with value: 0.5025376720370496 and parameters: {'n_estimators': 268, 'max_depth': 3, 'learning_rate': 0.06164174911746668, 'subsample': 0.961589020010445, 'c

Best params: {'n_estimators': 446, 'max_depth': 8, 'learning_rate': 0.08571905126785155, 'subsample': 0.8767209090036184, 'colsample_bytree': 0.9705548360819717, 'min_child_weight': 10, 'gamma': 0.627342729858057}
Best CV accuracy: 0.5206


In [15]:
# LOSO evaluation with fixed params
logo = LeaveOneGroupOut()
en_accuracies = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X_en, y_en, groups_en)):
    X_train, X_test = X_en[train_idx], X_en[test_idx]
    y_train, y_test = y_en[train_idx], y_en[test_idx]

    model = XGBClassifier(**best_params, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    en_accuracies.append(accuracy_score(y_test, preds))
    print(f"Fold {fold+1} | Accuracy: {en_accuracies[-1]:.4f}")

print("\n=== LOSO XGB (EN) ===")
print(f"Accuracy:  {np.mean(en_accuracies):.4f} ± {np.std(en_accuracies):.4f}")

Fold 1 | Accuracy: 0.5189
Fold 2 | Accuracy: 0.4825
Fold 3 | Accuracy: 0.4571
Fold 4 | Accuracy: 0.5661
Fold 5 | Accuracy: 0.4168
Fold 6 | Accuracy: 0.7442
Fold 7 | Accuracy: 0.8316
Fold 8 | Accuracy: 0.5769
Fold 9 | Accuracy: 0.6429
Fold 10 | Accuracy: 0.4201
Fold 11 | Accuracy: 0.6056
Fold 12 | Accuracy: 0.5352
Fold 13 | Accuracy: 1.0000
Fold 14 | Accuracy: 0.6571
Fold 15 | Accuracy: 0.4880
Fold 16 | Accuracy: 0.5459
Fold 17 | Accuracy: 0.4740
Fold 18 | Accuracy: 0.4545
Fold 19 | Accuracy: 0.3261
Fold 20 | Accuracy: 0.5784
Fold 21 | Accuracy: 0.5497
Fold 22 | Accuracy: 0.5263
Fold 23 | Accuracy: 0.3290
Fold 24 | Accuracy: 0.2192
Fold 25 | Accuracy: 0.6049
Fold 26 | Accuracy: 0.4333
Fold 27 | Accuracy: 0.5500
Fold 28 | Accuracy: 0.5610
Fold 29 | Accuracy: 0.6613
Fold 30 | Accuracy: 0.5079
Fold 31 | Accuracy: 0.4961
Fold 32 | Accuracy: 0.5070
Fold 33 | Accuracy: 0.4706
Fold 34 | Accuracy: 0.4715

=== LOSO XGB (EN) ===
Accuracy:  0.5356 ± 0.1396


In [16]:
# 10 Fold Cross CV with same tuned params
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores_10fold = cross_val_score(
    XGBClassifier(**best_params, n_jobs=-1),
    X_en, y_en,
    cv=cv_10fold,
    scoring='accuracy',
    n_jobs=-1
)

print("\n=== 10-FoldCV XGB (EN) ===")
print(f"Accuracy: {scores_10fold.mean():.4f} ± {scores_10fold.std():.4f}")


=== 10-FoldCV XGB (EN) ===
Accuracy: 0.6995 ± 0.0104
